# 05. XGBoost: Baseline vs Cost-Sensitive Learning

Notebook ini membandingkan model **XGBoost** tanpa dan dengan **cost-sensitive learning** (`sample_weight` balanced) pada dataset **Diabetes Health Indicators (BRFSS 2015)**.

Isi dan logika pelatihan mengikuti script sumber `train_xgboost.py`.

## 0. Mengatur Folder Kerja

Notebook ini disimpan di folder `notebooks`, sedangkan data dan artifacts berada di folder `glysense_backend`. Sel berikut memindahkan folder kerja satu tingkat ke atas agar jalur seperti `data/...` dan `artifacts/...` ditemukan. Aman dijalankan berulang.

In [1]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

d:\Tugas Akhir\Aplikasi Diabetes\glysense_backend


## 1. Membaca Dataset

Jalur file mengasumsikan notebook dijalankan dari direktori utama `glysense_backend`.

In [2]:
import sys
print(sys.executable)
print(sys.version)

d:\Tugas Akhir\Aplikasi Diabetes\glysense_backend\venv\Scripts\python.exe
3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]


In [3]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

file_path = "data/diabetes_012_health_indicators_BRFSS2015.csv"

df = pd.read_csv(file_path)

## 2. Memisahkan Fitur dan Target

In [5]:
X = df.drop(columns=["Diabetes_012"])

y = df["Diabetes_012"].astype(int)

## 3. Train-Test Split

Catatan: pembagian ini menggunakan `random_state=42` sesuai script asli, sehingga **berbeda** dari pembagian final penelitian (`random_state=2026`). Notebook ini bersifat eksperimen awal dan tidak menyimpan file apa pun.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## 4. XGBoost Tanpa Cost-Sensitive Learning

In [7]:
xgb_baseline = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_baseline.fit(X_train, y_train)

y_pred_baseline = xgb_baseline.predict(X_test)

In [8]:
print("=" * 70)
print("XGBOOST TANPA COST-SENSITIVE LEARNING")
print("=" * 70)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_baseline,
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_baseline))

XGBOOST TANPA COST-SENSITIVE LEARNING

Classification Report:
              precision    recall  f1-score   support

           0     0.8636    0.9787    0.9176     42741
           1     0.0000    0.0000    0.0000       926
           2     0.5622    0.1828    0.2759      7069

    accuracy                         0.8499     50736
   macro avg     0.4753    0.3872    0.3978     50736
weighted avg     0.8058    0.8499    0.8114     50736

Confusion Matrix:
[[41831     0   910]
 [  830     0    96]
 [ 5777     0  1292]]


## 5. Membuat Sample Weight untuk Cost-Sensitive Learning

In [9]:
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

## 6. XGBoost Dengan Cost-Sensitive Learning

In [10]:
xgb_csl = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_csl.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)

y_pred_csl = xgb_csl.predict(X_test)

In [11]:
print("=" * 70)
print("XGBOOST DENGAN COST-SENSITIVE LEARNING")
print("=" * 70)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_csl,
        digits=4,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_csl))

XGBOOST DENGAN COST-SENSITIVE LEARNING

Classification Report:
              precision    recall  f1-score   support

           0     0.9549    0.6404    0.7667     42741
           1     0.0293    0.2948    0.0533       926
           2     0.3430    0.6186    0.4413      7069

    accuracy                         0.6311     50736
   macro avg     0.4424    0.5180    0.4204     50736
weighted avg     0.8528    0.6311    0.7083     50736

Confusion Matrix:
[[27373  7408  7960]
 [  235   273   418]
 [ 1057  1639  4373]]
